In [1]:
import tensorflow as tf
import numpy as np 
import matplotlib.pyplot as plt

import random 
import os 
import re 
import string 
import tensorflow.data as tf_data
import keras 
import math 

vocab_size = 20000
maxlen = 90
embed_dim = 256 
batch_size = 128

# # Kết nối TPU thông qua cài đặt nếu cần thiết 
# resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
# # coonnection
# tf.config.experimental_connect_to_cluster(resolver)
# # khởi tạo 
# tf.tpu.experimental.initialize_tpu_system(resolver)
# # xây dựng môi trường phân tán 
# strategy = tf.distribute.experimental.TPUStrategy(resolver)

In [2]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  6969k      0  0:00:11  0:00:11 --:--:-- 8054k


In [3]:
batch_size = 128
import re
# The dataset contains each review in a separate text file
# The text files are present in four different folders
# Create a list all files
filenames = []
directories = [
    "aclImdb/train/pos",
    "aclImdb/train/neg",
    "aclImdb/test/pos",
    "aclImdb/test/neg",
]
for dir in directories:
    for f in os.listdir(dir):
        filenames.append(os.path.join(dir, f))

print(f"{len(filenames)} files")

# Create a dataset from text files
random.shuffle(filenames)
text_ds = tf_data.TextLineDataset(filenames)
text_ds = text_ds.shuffle(buffer_size=256)
text_ds = text_ds.batch(batch_size)


def custom_standardization(input_string):
    """Remove html line-break tags and handle punctuation"""
    lowercased = tf.strings.lower(input_string)
    stripped_html = tf.strings.regex_replace(lowercased, "<br />", " ")
    return tf.strings.regex_replace(stripped_html, f"([{string.punctuation}])", r" \1")



# Create a vectorization layer and adapt it to the text
vectorize_layer = keras.layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size - 1,
    output_mode="int",
    output_sequence_length=maxlen + 1,
)
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()  # To get words back from token indices


def prepare_lm_inputs_labels(text):
    """
    Shift word sequences by 1 position so that the target for position (i) is
    word at position (i+1). The model will use all words up till position (i)
    to predict the next word.
    """
    text = tf.expand_dims(text, -1)
    tokenized_sentences = vectorize_layer(text)
    x = tokenized_sentences[:, :-1]
    y = tokenized_sentences[:, 1:]
    return x, y


text_ds = text_ds.map(prepare_lm_inputs_labels, num_parallel_calls=tf_data.AUTOTUNE)
text_ds = text_ds.prefetch(tf_data.AUTOTUNE)



vocab = vectorize_layer.get_vocabulary()

def index_to_text(indexes):
    return " ".join([vocab[i] for i in indexes if i != 0])

for inputs, targets in text_ds.take(1):
    for i in range(5):  # In ra 5 mẫu đầu tiên từ batch
        input_text = index_to_text(inputs[i].numpy())
        target_text = index_to_text(targets[i].numpy())
        print(f"Input: {input_text}, Target: {target_text}")

50000 files
Input: elephant walk may not be the [UNK] of literature or of film , but it is great entertainment in the quasi [UNK] mode . it is the story of love , both genuine and illicit , as well as [UNK] ambition , devotion , and the arrogance of personal tyranny . a previous reviewer , john [UNK] , questions why the central focus of the film , the mansion called elephant walk , should have been built by the former owner , the [UNK] " the late tom [UNK] ,, Target: walk may not be the [UNK] of literature or of film , but it is great entertainment in the quasi [UNK] mode . it is the story of love , both genuine and illicit , as well as [UNK] ambition , devotion , and the arrogance of personal tyranny . a previous reviewer , john [UNK] , questions why the central focus of the film , the mansion called elephant walk , should have been built by the former owner , the [UNK] " the late tom [UNK] , right
Input: while this is horribly dated , i must insist . . .please , no remake ! frankly ,

Text Generator

In [4]:
# Xây dụng lớp sinh văn bản cho GPT 
# được khởi tạo như là 1 callback
class TextGenerator(keras.callbacks.Callback):
    """
    Một phương thức gọi lại để sinh văn bản từ mô hình được huấn luyện 
        1. Cung cấp một số lời nhắc bắt đầu cho mô hình 
        2. Dự đoán xác xuất cho token tiếp theo. 
        3. Lấy mẫu mã thông báo tiếp theo và thêm nó vào đầu vào tiếp theo 

    Argumnets: 
        Max_tokens : Integer , số lượng các tokens được tạo lời nhắc (tối đa)
        Start_token : List of integers , các chỉ số mã thông báo cho lời nhắc bắt đầu 
        Text_to_word : List of string , Là kết quả từ lớp TextVectorization layer 
        Top_k : Integẻ , lấy mâix từ top_k là token được dự đoán
    """
    # Thiết lập phương phức khởi tạo và định nghĩa các tham số 
    # max_token (số tokens đối đa được sinh ra)
    # star_tokens (chỉ số mã thông báo cho lời nhắc)
    # Index_to_word (từ điển )
    def __init__(
        self, max_tokens, start_tokens, index_to_word, top_k=10, print_every=1
    ):
        self.max_tokens = max_tokens
        self.start_tokens = start_tokens
        self.index_to_word = index_to_word
        self.print_every = print_every
        self.k = top_k

    # Thiết lập phương thức lấy mẫu nhận đầu vào là 1 vector shape = (vocab_size,)
    # từ kết quả đầu ra của dự đoán 
    def sample_from(self, logits):
        # lấy ra giá trị và chỉ số của tokens theo top_k 
        logits , indices = tf.math.top_k(logits , k=self.k, sorted=True)
        # Biến đổi các chỉ số thành ma trận 1 chiều bằng np.assarray
        indices = np.asarray(indices).astype("int32")
        # Dự đoán xác xuất của các tokens dựa vào ma trận giá trị logits 
        # Thêm vào chiều thứ nhất cho ma trận logits => shape [1 , vocab_size] và bỏ đi chiều đầu
        # của tensor logits kết quả là 1 ma trận shape = [vocab_size,]
        preds = keras.activations.softmax(tf.expand_dims(logits ,0))[0]
        # biến đổi pred thành ma trận 
        preds = np.asarray(preds).astype("float32")
        # Trả về ma trận indices và các phần tử trong ma trận đuược chọn ngẫu nhiên 
        # với xác xuất được xác định bởi preds
        return np.random.choice(indices, p=preds)
    
    # Xây dựng phương thức giải mã token 
    # Nhận đầu vào  number số lượng tokens
    # lấy ra tokens từ tập từ điển bằng cachs ánh xạ các chỉ số qua tập từ điển 
    def detokenize(self, number) :
        return self.index_to_word[number]

    # Thiết lập phương thức def on epochs 
    def on_epoch_end(self, epoch , logs=None):
        # tạo một danh sách start_token bằng cách sao chép các phần tử từ dnah sách 
        # self.start_tokens là 1 danh sách chứa các tokens  interger đã cho . 
        start_tokens = [_ for _ in self.start_tokens]
        # Kiểm tra xem epochs hiện tại có chia hết cho thuộc tinh self.prin_every khôn
        # print_every là một số nguyên nếu không nghĩa là kết thúc phương thưc 0 sinh văn bản 
        if (epoch + 1) % self.print_every !=0 :
            return 
        
        # Tạo danh sách dỗng để lưu chữ số tokens được sinh ra và đếm số tokens được sinh 
        num_tokens_generated = 0
        tokens_generated = []
        # sử dụng 1 vòng wihle và kiểm tả điều kiện khi số tokens được sinh ra còn nhỏ hơn 
        # max_tokens 
        while num_tokens_generated <= self.max_tokens:
            # Lấy ra pad_len là độ dài đệm cần thêm cho danh sách start_tokens 
            pad_len = maxlen- len(start_tokens)
            # gán giá trị của biến sample bằng chỉ số của start_token - 1
            sample_index = len(start_tokens) - 1
            # kiểm tra điều kiện nếu pad_len < 0 : kiểm tra xem độ dài cần thêm vào < 0    
            # nghĩa là danh sách start token vượt quá độ dài tối da 
            if pad_len < 0:
                # gán giá trị của x bằng 1 dnah sách con của start_token đến max_len- 1
                x = start_tokens[:maxlen]
                # gán sample_index = chỉ số cuối cùng trong danh sách len x
                sample_index = maxlen - 1
            # nếu pad_ > 0  là độ dài cần thêm chưa đạt 
            elif pad_len > 0:
                # gán giá trị của x bằng cách kết hợp với pad_token chứa các chỉ số =  0
                x = start_tokens + [0]* pad_len
            # trường hợp còn lại 
            else: 
                # Gán trực tiếp x =  start_tokens 
                x = start_tokens 
            # Biến đổi x Thanh mảng numpy  có hai chiều 
            x = np.asarray([x])
            # dự đoán kết quả từ  x bằng mô hình và trả về 2 giá trị y là xác xuất 
            # cho từng tokens và _ là trạng thái ẩn của mô hình y shape = [1 , maxlen , vocab_size] với maxlen cột và mỗi cột biểu diễn vocab_size giá trị 
            y , _ = self.model.predict(x, verbose=0)
            #  Lấy mẫu một token ngẫu nhiên từ vector xác suất tại vị trí [0][sample_index] trong ma trận y 
            # Trả về 1 token ngẫu nhiên và gán nó vào sample_tokens (là danh sách chưá các giá trị tokens được sinh ra dựa trên xác suất của chúng) 
            # y[0][sample_index] shape=  (vocab_size,)  với sample_index để tính toán cho tokens theo chỉ số tokens tiếp theo cần được sinh ra
            sample_token = self.sample_from(y[0][sample_index])
            # Thêm sample tokens là các tokens ngẫu nhiên có thể được sinh ra từ danh sachs token ngẫu nhiên
            tokens_generated.append(sample_token)
            # Thêm toke ngẫu nhiên đã được sinh ra vào danh sách start tokens danh sách này 
            # dùng để cập nhật lại chuỗi tokens bắt đầu để sinh ra tokens tiếp theo 
            start_tokens.append(sample_token)
            # cập nhật lại số lượng tokens được sinh ra bằng cách tính toán lại lượng tokens đã được sinh ra 

            num_tokens_generated = len(tokens_generated)
        # Tạo ra một chuỗi văn bản hoàn chỉnh từ các token được sinh ra
        # 1 Nối danh sách start_tokens và danh sách token_generated lại thành 1 danh sách duy nhất 
        # chứa tất cả các tokens đã được sinh ra 
        # 2 Chuyển đổi mỗi tokens thành 1 từ hoặc ký tự tương ứng phép mã hóa ngược để sinh văn bản gốc
        # 3 Sử dụng join để nối các từ hay các ký tự thành 1 chuỗi văn bản cách 1  khoảng trăng 
        txt = " ".join(
            [self.detokenize(_) for _ in self.start_tokens + tokens_generated]
        )
        # Hiển thị ra đoạn văn được GPT tạo ra với 40 tokens
        print(f"generated text:\n {txt} \n")

# Mã hóa lời nhắc 
# Tạo một từ điển rỗng để lưu trữ các từ và chỉ số của từ trong từ điển 
word_to_index = {}
for index , word in enumerate(vocab):
    word_to_index[word] = index

# Tạo một chuỗi lời nhắc 
start_prompt = "this movie is"
# tạo ra một danh sách start_tokens để lưu trữ các giá trị tokens từ start_prompt 
# sử dụng hàm get của từ điển word_to_index để lấy ra giá trị tương ứng với khóa là biến _ 
# là chỉ số của từ đos trong từ điển nếu không có khóa gán = 1
start_tokens = [word_to_index.get(_,1) for _ in start_prompt.split()]
num_tokens_generated = 45
# đưa các tham số vào trong call back 
text_gen_callback = TextGenerator(num_tokens_generated, start_tokens , vocab)

In [5]:
def attention_mask(sequence_length, batch_size):

  # shape tensor = [seqlength, None]
  i = tf.range(sequence_length)[:, None]
  # shape tensor = [seqlength]
  j = tf.range(sequence_length)

  # shape m = [sequence_length, sequence_length]
  # phép tính này sẽ mở rộng tensor i [seq_length, 1] ->i có kích thước [n_dest, 1],
  # sẽ được "mở rộng" thành [n_dest, n_src] để phù hợp với kích thước của j.
  m = i >= j - sequence_length + sequence_length
  mask = tf.cast(m, dtype="bool") # boolean tensor
  mask = tf.reshape(mask, [1, 1, sequence_length, sequence_length])

  # sử dụng hàm title để nhân tensor mask với 1 tensor có shape được chỉ định
  mask = tf.tile(mask, [batch_size, 1, 1, 1]) # Tile to match batch size
  return tf.cast(mask, tf.float32)

In [6]:
class RMSNorm(keras.layers.Layer):
    def __init__(self, units, epsilon=1e-6, **kwargs):
        super(RMSNorm, self).__init__(**kwargs)
        self.units = units
        self.epsilon = epsilon
        self.gamma = None

    def build(self, input_shape):
        self.gamma = self.add_weight(
            name="gamma",
            shape=(self.units,),
            initializer="ones",
            trainable=True,
        )

    def _norm(self, x):
        # Calculate the mean and variance of the input tensor
        mean = tf.reduce_mean(x, axis=-1, keepdims=True)
        variance = tf.reduce_mean(tf.square(x - mean), axis=-1, keepdims=True)
        # Normalize the input tensor
        return (x - mean) / tf.sqrt(variance + self.epsilon)

    def call(self, x):
        # Apply the normalization and scaling
        return self.gamma * self._norm(tf.cast(x, tf.float32)) # Cast the input tensor to float32

In [7]:
# Xây dựng phương thức nhúng rotary embedding 
def precompute_theta_pos_frequenceis(head_dim, sequence_length, theta=10000.0):
  # tạo một tensor thetanumber shape = head_dim / 2 
    theta_number = tf.range(0, head_dim, 2, dtype=tf.float32)
  # Tính toán 1 góc quay theta 
    theta_number = 1.0 * (theta ** (theta_number / head_dim))
  # khởi tạo tensor m shape = sequence_length tensor này được sử dụng để mở rộng tensor freqs
    m = tf.range(sequence_length, dtype=tf.float32)

   # Multiply each theta by each position using the outer product.
   # Shape: (Seq_Len) outer_product (Head_Dim / 2) -> (Seq_Len, Head_Dim / 2)
    freqs = tf.tensordot(m, theta, axes=0)

    # Compute complex numbers in polar form
    # Shape: (Seq_Len, Head_Dim / 2)
    freqs_complex = tf.complex(tf.ones_like(freqs), freqs)
    return freqs_complex


def apply_rotary_embedding(x, freqs_complex):
    # Separate the last dimension pairs of two values representing the real and imaginary parts of the complex number
    # shape [batch_size, seq_length, head_dim] -> [batch_size, seq_length, head_dim / 2]
    x_complex = tf.complex(x[..., ::2], x[..., 1::2])

    # Reshape the freqs_complex tensor to match the shape of the x_complex tensor
    # shape [seq_length, head_dim / 2] -> shape [1, seq_length, 1, head_dim / 2]
    freqs_complex = tf.expand_dims(tf.expand_dims(freqs_complex, axis=0), axis=2)

    # Multiply each complex number in the x_complex tensor by the corresponding complex number in the freqs_complex tensor
    # (B, Seq_Len, H, Head_Dim/2) -> (B, Seq_Len, H, Head_Dim/2, 2)
    x_rotated = x_complex * freqs_complex

    # Convert the complex number back to the real number
    # (B, Seq_Len, H, Head_Dim/2, 2) -> (B, Seq_Len, H, Head_Dim)
    x_out = tf.concat([tf.math.real(x_rotated), tf.math.imag(x_rotated)], axis=-1)

    return x_out

In [14]:
class Attention(keras.layers.Layer):
    def __init__(self, num_head, embed_dim):
      super(Attention, self).__init__()
      self.num_head = num_head
      self.embed_dim = embed_dim
      self.head_dim = embed_dim // num_head

      self.cal = (embed_dim // num_head) ** -0.5


      self.wq = keras.layers.Dense(embed_dim, kernel_initializer="glorot_uniform")
      self.wk = keras.layers.Dense(embed_dim, kernel_initializer="glorot_uniform")
      self.wv = keras.layers.Dense(embed_dim, kernel_initializer="glorot_uniform")

      self.wo = keras.layers.Dense(embed_dim)

      self.dropout = keras.layers.Dropout(0.1)

    def split_head(self, x, batch_size):
      # reshape tenssor [batch_size, seq_length, num_head, head_dim]
      x = tf.reshape(x, (batch_size, tf.shape(x)[1], self.num_head, self.head_dim)) 
      # transspose tenssor [shape [batch_size, num_head, seq_length, head_dim]]
      return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs):

      # inputs shape [batch_size, seq_length, embed_dim]
      batch_size = tf.shape(inputs)[0]
      sequence_length = tf.shape(inputs)[1]
      # create Q, K, V vector 
      Q = self.wq(inputs)
      K = self.wk(inputs)
      V = self.wv(inputs)

      # reshape tensor shape 
      # batch_size, num_head, sequence_length, head_dim 
      Q = self.split_head(Q, batch_size)
      K = self.split_head(K, batch_size)
      V = self.split_head(V, batch_size)
        
        
      # K = K * self.cal 
      # apply rotary embedding for q, k vector 
      freqs_complex = precompute_theta_pos_frequenceis(self.head_dim, sequence_length)

      Q = apply_rotary_embedding(Q, freqs_complex)
      K = apply_rotary_embedding(K, freqs_complex)

      # Tính toán attention q.K 
      # shape [batch_size, num_head, sequence_length, sequence_length]
      # shape Q = [batch_size, num_head, seq_length, head_dim] * shape [batch_size, num_head, head_dim, sequence_length]
      attention_score = tf.matmul(Q, K, transpose_b=True) 
      # attention / dk 
      d = tf.cast(self.head_dim, tf.float32)
      score = attention_score / tf.math.sqrt(d)
    
      # compute attention mask 
#       mask = attention_mask(sequence_length, batch_size)
#       if mask is not None: 
#         score += mask * 1e-9 
      # bỏ qua việc áp dụng attention mask  áp dụn trực tiếp hàm softmax 
      attention_weights = tf.nn.softmax(score, axis=-1)
      # attention_weights = self.dropout(attention_weights)
      
      # nhân vector attention với v
      # shape tensor = batch_size, num_head, seq_length, seq_length * batch_size, num_head, seq_length, embed_dim - >
      # shape = batch_size, num_head, seq_length, embed_dim
      attention_ = tf.matmul(attention_weights, V)
      # transpose shape = batch_size, seqence_length, num_head, head_Dim
      attention_ = tf.transpose(attention_, perm=[0, 2, 1, 3])
      # reshape tensor -> shape [batch_size, seq_length, embed_dim]
      attention_ = tf.reshape(attention_, (batch_size, -1, self.embed_dim))

      output = self.wo(attention_)
      return output


In [15]:
class Feedforward(keras.layers.Layer):
  def __init__(self, embed_dim, **kwargs):
    super(Feedforward, self).__init__(**kwargs)
    self.embed_dim = embed_dim
    self.feedfor_dim = embed_dim  * 2

    # lớp ẩn thứ nhất sẽ có kích thước đâu ra là ff_dim 
    self.dense1 = keras.layers.Dense(self.feedfor_dim)  
    self.dense2 = keras.layers.Dense(self.embed_dim)
    self.dense3 = keras.layers.Dense(self.feedfor_dim) 
    
  def call(self, x):
    # sử dụng hàm silu lên vector đầu vào 
    # x = [ batch_size, sequence_length, embed_dim ] -> [batch_size, sequence_length, ff_fim]
    x_q = tf.nn.silu(self.dense1(x))
    # tính toán lớp ẩn thứ 2 
    # x_out = [batch_size, sequence_length, embed_dim] -> [batch_size, sequence_length, ff_fim]
    X_out = self.dense3(x)
    #  Nhân x_q  với x_out 
    # shape [batch_size, seq_length, ff_dim]
    output = x_q * X_out
    # nhúng ouput qua dense 2 shape [batch_size, sequence_length, ff_dim] -> [batch_size, seq_length, embed_dim]
    x = self.dense2(output)
    return x


In [16]:
class Encoder(keras.layers.Layer):
  def __init__(self, embed_dim, num_head, **kwargs):
    super(Encoder, self).__init__(**kwargs)
    self.embed_dim = embed_dim
    self.num_head = num_head
    self.head_dim = embed_dim // num_head

    self.attention = Attention(num_head, embed_dim)
    self.feedforward =  Feedforward(embed_dim) 
    self.RMSNorm = RMSNorm(embed_dim)
    self.RMSNorm2 = RMSNorm(embed_dim)

  def call(self, x):
    # compute attention 
    attention = self.attention(x)
    # layer RMSNorm 
    Rms = self.RMSNorm(x + attention)
    # FFN layer 
    output = self.feedforward(Rms)
    output = self.RMSNorm2(Rms + output)
    return output

In [17]:
# Xây dựng mô hình Transformer
class Transformer(keras.layers.Layer):
  # Thiết lập phương thức khởi tạo vào định nghĩa các thuộc tính
  def __init__(self, num_layers, embed_dim, num_heads):
    super().__init__()
    # Định nghĩa các thuộc tính
    self.num_layers = num_layers
    # embedding_dim
    self.embed_dim = embed_dim
    self.Token_embedding = keras.layers.Embedding(vocab_size, embed_dim)
    # định nghĩa layer là một module list
    # **Sửa lỗi:** Tạo một đối tượng `keras.Sequential`
    self.layers = keras.Sequential([
       Encoder(embed_dim, num_heads)
       for _ in range(num_layers)
    ])


    self.RMSNorm = RMSNorm(embed_dim)
    self.Linear = keras.Sequential([
         # keras.layers.Dense(embed_dim, activation="relu"),
         keras.layers.Dense(embed_dim),
    ])
  def call(self, inputs):
    # apply token_embedding 
    inputs = self.Token_embedding(inputs)
    x = inputs 
    # apply transformer Encoder layers 
    for layer in range(self.num_layers):
       inputs = self.layers(inputs)
    # Layer Root Mean Square Norm 
    out = self.RMSNorm(inputs)
    # linear Projection layer 
    outputs = self.Linear(out)
    return outputs

In [18]:
# Create model 
def create_model(num_layers, maxlen, vocab_size, embed_dim, num_heads,rate=0.1):
  # Tạo lớp input layer 
  inputs = keras.layers.Input(shape=(maxlen,), dtype=tf.int32)
  # Transformer model 
  transformer = Transformer(num_layers, embed_dim, num_heads)
  # Thực hiênhj nhúng ngữ cảnh transformer 
  x = transformer(inputs)

  # Thực hiện một lớp nhúng tuyến tính
  outputs = keras.layers.Dense(vocab_size)(x)
  #
  #outputs = keras.layers.Dense(vocab_size, activation='softmax')(x)
  #
  model = keras.Model(inputs=inputs, outputs=[outputs, x])

  # Sử dụng trình tối ưu hóa AdamW 
  # optimizer = keras.optimizers.Adam(learning_rate=1e-4)
  # optimizer = tf.keras.optimizers.Adam()
  loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
  model.compile(optimizer="Adam", loss=[loss_fn, None])
  
  return model


In [19]:
# # Tóm tắt mô hình để có thể xem được thông số mô hình
# with  strategy.scope():
model = create_model(
    num_layers=1,
    maxlen = 90,
    vocab_size= 20000,
    embed_dim=256,
    num_heads=4,
    # feed_forward_dim=256,
)

model.summary()
model.fit(text_ds, verbose=2, epochs=10, callbacks=[text_gen_callback])


/opt/conda/lib/python3.10/site-packages/keras/src/layers/layer.py:361: UserWarning: `build()` was called on layer 'attention_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 90)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_1 (Transformer)     │ (None, 90, 256)        │     5,844,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 90, 20000)      │     5,140,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,984,224 (41.90 MB)

 Trainable params: 10,984,224 (41.90 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


I0000 00:00:1725119361.437883    9078 service.cc:145] XLA service 0x7d793003d690 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1725119361.437952    9078 service.cc:153]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1725119361.437957    9078 service.cc:153]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
W0000 00:00:1725119361.751437    9078 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1725119368.099004    9078 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
W0000 00:00:1725119430.609968    9079 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
I0000 00:00:1725119436.785002    9144 asm_compiler.cc:369] ptxa

generated text:
 this movie is a very bad this movie and a few years after the movie to be a [UNK] is one of the movie was one . it 's not worth the best , this movie . the best .          

391/391 - 99s - 253ms/step - loss: 5.4922
Epoch 2/10
generated text:
 this movie is a great movie is the first , but it was a great , i don 't get better , it has a big names of the movie .                   

391/391 - 66s - 169ms/step - loss: 4.9828
Epoch 3/10
generated text:
 this movie is .                                              

391/391 - 65s - 166ms/step - loss: 4.8646
Epoch 4/10
generated text:
 this movie is the movie , and his role as a [UNK] . i love this movie . the movie , and [UNK] , and the best role is a movie .                  

391/391 - 66s - 168ms/step - loss: 4.7670
Epoch 5/10
generated text:
 this movie is very little too bad acting and i think about a lot of the worst movie . [UNK] !                             

391/391 - 66s - 169ms/step - loss: 4.6909
Epoch 6/10
gen